# Program *Training*: Jaringan Saraf Tiruan dengan *Back Propagation*

Mengenali 3 kelas kendaraan (pesawat, mobil, kapal) dari citra *grayscale* (abu-abu, satu nilai terang tiap *pixel*) 32 x 32.
Algoritma dan notasi mengikuti Fausett, *Fundamentals of Neural Networks* (1994), Bab 6, Step 0 sampai 9.

Hasil akhir: satu berkas `BOBOT.npz` berisi `v`, `v_0`, `w`, `w_0` (dan ukuran gambar), ditambah versi `BOBOT.zip` berisi 5 berkas Excel (`.xlsx`) yang rapi dan bisa dibaca manusia. Keduanya diunduh otomatis ke komputer.

Jalankan sel dari atas ke bawah. Sel 5 sampai 10 memperlihatkan tiap step pada satu gambar contoh. Sel 11 melatih jaringan dengan seluruh data latih.

## Sel 1. Pengaturan Awal dan Konfigurasi Jaringan

* Menyambungkan Google Colab ke Google Drive serta menetapkan alamat folder (*path*) *dataset* dan berkas luaran bobot (`BOBOT.npz`).
* Menetapkan arsitektur jaringan: 1024 *node input* (citra $32 \times 32$), 100 *node hidden* (lapisan tengah), dan 3 *node output* (pesawat, mobil, kapal).
* Mengatur *hyperparameter* (pengaturan yang ditetapkan sebelum *training*):
   * *learning rate* (`alpha` = 0.1): besar langkah koreksi bobot tiap pembaruan.
   * *epoch* maksimum 200 (*epoch*: satu putaran penuh melewati seluruh data latih).
   * ambang galat 0.01 (galat: selisih keluaran jaringan dan target).
   * rentang bobot awal acak dan *random seed* (angka awal pengacak agar hasil bisa diulang).
* **Analogi:** pelatih menyiapkan ruang kerja tim penyortir foto: berapa pengamat (1024), berapa ahli (100), berapa juru bicara (3), seberapa keras pelatih menegur (*learning rate*), serta kapan latihan berhenti.

In [ ]:
import os
import time
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import drive
from IPython.display import display, Math   # Math: menampilkan persamaan LaTeX

drive.mount('/content/drive')

# Lokasi di Google Drive
FOLDER_PROYEK = '/content/drive/MyDrive/TUGAS_ALGORITMA_AI'
FOLDER_LATIH  = FOLDER_PROYEK + '/DATASET/LATIH'   # gambar untuk belajar
FOLDER_UJI    = FOLDER_PROYEK + '/DATASET/UJI'     # gambar untuk menguji, tidak ikut dilatih
BERKAS_BOBOT  = FOLDER_PROYEK + '/BOBOT/BOBOT.npz' # tempat menyimpan hasil training

# Ukuran gambar dan arsitektur
LEBAR, TINGGI = 32, 32  # ukuran gambar dalam pixel (titik gambar)
n = LEBAR * TINGGI      # n: jumlah node input, satu per pixel
p = 100                 # p: jumlah node hidden, lapisan tengah
m = 3                   # m: jumlah node output, satu per kelas

# Parameter pembelajaran
alpha        = 0.1      # learning rate: besar langkah koreksi bobot
EPOCH_MAKS   = 200      # epoch: satu putaran seluruh data latih; batas maksimum
AMBANG_GALAT = 0.01     # galat: selisih keluaran dan target; berhenti bila rata-rata di bawah ini
BATAS_ACAK   = 0.5      # bobot awal diacak antara -0,5 dan 0,5
BIJI_ACAK    = 42       # random seed: angka awal pengacak agar hasil bisa diulang
CETAK_TIAP   = 10       # cetak laporan tiap 10 epoch

# Urutan kelas menentukan target: indeks 0 = 1 0 0, indeks 1 = 0 1 0, indeks 2 = 0 0 1
NAMA_KELAS = ['KELAS1_PESAWAT', 'KELAS2_MOBIL', 'KELAS3_KAPAL']

print('=' * 52)
print('PENGATURAN')
print('=' * 52)
print(f'Node input  (n) : {n}  ({LEBAR} x {TINGGI} pixel)')
print(f'Node hidden (p) : {p}')
print(f'Node output (m) : {m}')
print(f'alpha           : {alpha}')
print(f'Epoch maksimum  : {EPOCH_MAKS}')
print(f'Ambang galat    : {AMBANG_GALAT}')

## Sel 2. Prapemrosesan Citra dan Pemuatan *Dataset*

* Fungsi `praproses` mengubah satu gambar menjadi vektor masukan dalam 4 langkah:
   * *grayscale*: ubah warna menjadi satu nilai terang tiap *pixel*.
   * pengaman ukuran: bila ukuran bukan $32 \times 32$, ubah ukurannya (*resize*).
   * *flatten*: ubah gambar 2 dimensi menjadi satu baris berisi 1024 nilai.
   * normalisasi: bagi dengan 255.0 agar nilai berada pada skala 0 hingga 1.
* Fungsi `muat_dataset` memuat seluruh data latih dan uji, lalu memetakan label target ke format *one-hot* (hanya satu *node* bernilai 1, sisanya 0): `1 0 0`, `0 1 0`, atau `0 0 1`.
* **Analogi:** pelatih menyiapkan tumpukan kartu foto. Tiap foto dibuat hitam putih dengan ukuran sama, lalu kunci jawabannya dicatat dari nama folder tempat foto itu disimpan.

In [ ]:
def praproses(gambar):
    """Ubah satu gambar PIL (RGB) menjadi vektor x berisi n nilai, rentang 0 sampai 1."""
    abu = gambar.convert('L')                        # 1. grayscale: satu nilai terang per pixel
    if abu.size != (LEBAR, TINGGI):                  # 2. pengaman ukuran: ubah ke 32 x 32 bila beda
        abu = abu.resize((LEBAR, TINGGI))
    x = np.asarray(abu, dtype=np.float64).flatten()  # 3. flatten: gambar 2 dimensi jadi satu baris
    return x / 255.0                                 # 4. normalisasi: nilai menjadi 0 sampai 1


def muat_dataset(folder):
    """Baca semua gambar di folder. Hasil: X (data), T (target), label (nomor kelas), nama berkas."""
    X, T, label, nama_berkas = [], [], [], []
    for k, nama in enumerate(NAMA_KELAS):
        daftar = sorted(f for f in os.listdir(f'{folder}/{nama}') if f.endswith('.png'))
        for berkas in daftar:
            X.append(praproses(Image.open(f'{folder}/{nama}/{berkas}')))
            target = np.zeros(m)
            target[k] = 1.0                          # one-hot: hanya node kelas ini bernilai 1
            T.append(target)
            label.append(k)                          # label: nomor kelas, untuk menghitung akurasi
            nama_berkas.append(f'{nama}/{berkas}')
    return np.array(X), np.array(T), np.array(label), nama_berkas


X_latih, T_latih, label_latih, nama_latih = muat_dataset(FOLDER_LATIH)
X_uji,   T_uji,   label_uji,   _          = muat_dataset(FOLDER_UJI)

print('=' * 52)
print('DATASET DIMUAT')
print('=' * 52)
print(f'{"Kelas":<18}{"Latih":>8}{"Uji":>8}   Target')
for k, nama in enumerate(NAMA_KELAS):
    target = ' '.join(str(int(a)) for a in np.eye(m)[k])
    print(f'{nama:<18}{(label_latih == k).sum():>8}{(label_uji == k).sum():>8}   {target}')
print('-' * 52)
print(f'{"Total":<18}{len(X_latih):>8}{len(X_uji):>8}')
print(f'\nBentuk X_latih : {X_latih.shape}  (pola x node input)')
print(f'Rentang nilai  : {X_latih.min():.2f} sampai {X_latih.max():.2f}')

## Sel 3. Verifikasi Konversi Citra

* Menampilkan perbandingan visual antara gambar asli berwarna di baris atas dengan hasil olah *grayscale* di baris bawah untuk memvalidasi masukan model.
* **Analogi:** pelatih memeriksa beberapa kartu foto sebelum latihan: bentuk foto masih sama, hanya warnanya yang hilang.

In [ ]:
# baris atas: gambar asli berwarna; baris bawah: hasil praproses (grayscale)
fig, sumbu = plt.subplots(2, 3, figsize=(7, 5))
for k, nama in enumerate(NAMA_KELAS):
    berkas = sorted(f for f in os.listdir(f'{FOLDER_LATIH}/{nama}') if f.endswith('.png'))[0]
    asli = Image.open(f'{FOLDER_LATIH}/{nama}/{berkas}')
    sumbu[0, k].imshow(asli)
    sumbu[0, k].set_title(nama.split('_')[1] + f'\n(mode {asli.mode})')
    sumbu[1, k].imshow(praproses(asli).reshape(TINGGI, LEBAR), cmap='gray', vmin=0, vmax=1)
    sumbu[1, k].set_title('grayscale')
    sumbu[0, k].axis('off'); sumbu[1, k].axis('off')
plt.tight_layout()
plt.show()

## Sel 4. Fungsi Aktivasi dan Inisialisasi Bobot Awal (Step 0)

* Menyiapkan fungsi aktivasi sigmoid `f(x) = 1 / (1 + e^-x)` (mengubah nilai apa pun menjadi 0 sampai 1) dan turunannya `f'(x) = f(x)(1 - f(x))`.
* Mengisi seluruh bobot dan *bias* (nilai tambahan tetap tiap *node*) secara acak seragam antara $-0.5$ hingga $0.5$ sebagai tebakan mula-mula model.
* *Output* sel diberi label **Step 0** (*Initialize weights*) seperti di buku.
* **Analogi:** hari pertama, tim belum tahu apa-apa. Tingkat percaya dan kecenderungan bawaan tiap orang diisi acak kecil. Lampu (sigmoid) hanya bisa menyala antara 0 (padam) dan 1 (terang penuh).

In [ ]:
def f(x):
    """Fungsi aktivasi sigmoid biner: f(x) = 1 / (1 + e^-x)."""
    return 1.0 / (1.0 + np.exp(-x))

def f_aksen(x):
    """Turunan sigmoid: f'(x) = f(x) (1 - f(x))."""
    return f(x) * (1.0 - f(x))


# Step 0: inisialisasi bobot dengan bilangan acak kecil
rng = np.random.default_rng(BIJI_ACAK)               # rng: pembangkit bilangan acak
v   = rng.uniform(-BATAS_ACAK, BATAS_ACAK, (n, p))   # v: bobot input ke hidden
v_0 = rng.uniform(-BATAS_ACAK, BATAS_ACAK, p)        # v_0: bias node hidden
w   = rng.uniform(-BATAS_ACAK, BATAS_ACAK, (p, m))   # w: bobot hidden ke output
w_0 = rng.uniform(-BATAS_ACAK, BATAS_ACAK, m)        # w_0: bias node output

print('=' * 62)
print('Step 0 : Initialize weights (set to small random values)')
print('=' * 62)
print(f'{"Simbol buku":<13}{"Nama di kode":<14}{"Ukuran":<14}{"Jumlah nilai":>12}')
for simbol, nama, larik in [('v_ij', 'v', v), ('v_0j', 'v_0', v_0), ('w_jk', 'w', w), ('w_0k', 'w_0', w_0)]:
    print(f'{simbol:<13}{nama:<14}{str(larik.shape):<14}{larik.size:>12,}')
print(f'\nRentang bobot awal: {min(v.min(), w.min()):.2f} sampai {max(v.max(), w.max()):.2f}')

## Sel 5. Step 3: Pola Contoh dan Sinyal *Input*

* Memilih gambar latih pertama sebagai **pola contoh**. Sel 5 sampai 10 memakai pola ini agar tiap step dapat dilihat satu per satu.
* Step 3: setiap *input unit* $X_i$ menerima sinyal $x_i$ (nilai *pixel* ternormalisasi), lalu meneruskannya ke semua *hidden unit*.
* **Analogi:** satu kartu foto diperlihatkan. Tiap dari 1024 pengamat melaporkan kecerahan satu titik fotonya, lalu laporan itu dikirim ke semua ahli.

In [ ]:
x = X_latih[0]    # x: sinyal input, nilai 1024 pixel pola contoh
t = T_latih[0]    # t: target kelas pola contoh

# gambar yang sedang dibaca: 32 x 32 pixel, satu nilai terang per pixel
plt.figure(figsize=(2.5, 2.5))
plt.imshow(x.reshape(TINGGI, LEBAR), cmap='gray', vmin=0, vmax=1)
plt.title('Pola contoh (grayscale)')
plt.axis('off')
plt.show()

print('=' * 62)
print('Step 3 : Input unit X_i menerima x_i, dikirim ke semua hidden unit')
print('=' * 62)
print(f'Pola contoh : {nama_latih[0]}')
print(f'Target t    : {" ".join(str(int(a)) for a in t)}  (kelas {NAMA_KELAS[int(np.argmax(t))].split("_")[1].lower()})')
print()
display(Math(r'x_i = \frac{\text{nilai pixel ke-}i}{255}'))
print('Keterangan:')
print('  x_i adalah kecerahan pixel ke-i: 0 = hitam, 1 = putih')
print(f'  Gambar punya {TINGGI} baris, tiap baris berisi {LEBAR} pixel, jadi n = {TINGGI} x {LEBAR} = {n}')
print(f'  Urutan pixel: dari kiri atas, baris demi baris. Baris gambar 1 = x_1 sampai x_{LEBAR}, baris 2 = x_{LEBAR + 1} sampai x_{2 * LEBAR},')
print(f'  dan seterusnya sampai baris {TINGGI} = x_{n - LEBAR + 1} sampai x_{n} (pojok kanan bawah)')
print(f'  Satu baris gambar dicetak dalam {LEBAR // 8} baris angka, masing-masing 8 pixel')
print()
for r in range(TINGGI):
    print(f'Baris gambar {r + 1:>2}  (x_{r * LEBAR + 1} sampai x_{(r + 1) * LEBAR})')
    for awal in range(r * LEBAR, (r + 1) * LEBAR, 8):
        print(f'  x_{awal + 1} sampai x_{awal + 8}'.ljust(24) + ' '.join(f'{a:.2f}' for a in x[awal:awal + 8]))

## Sel 6. Step 4: Keluaran *Hidden Layer*

$$z\_in_j = v_{0j} + \sum_{i=1}^{n} x_i v_{ij}, \qquad z_j = f(z\_in_j)$$

* Fungsi `langkah_4` mengerjakan rumus di atas untuk tiap *node hidden* $j = 1, \dots, p$. Jumlah atas $i$ dikerjakan `np.sum`, karena perulangan $i$ satu per satu terlalu lambat (sekitar 35 detik per *epoch* hanya untuk Step 4).
* **Analogi:** tiap ahli menimbang laporan 1024 pengamat dengan tingkat percayanya sendiri, menambahkan kecenderungan bawaannya, lalu menyalakan lampunya (0 sampai 1).

In [ ]:
def langkah_4(x, v, v_0):
    """Step 4: z_in_j = v_0j + SUM_i x_i v_ij, lalu z_j = f(z_in_j)."""
    z_in = np.zeros(p)
    z    = np.zeros(p)
    for j in range(p):
        z_in[j] = v_0[j] + np.sum(x * v[:, j])   # np.sum: jumlahkan semua suku i
        z[j]    = f(z_in[j])
    return z_in, z


z_in, z = langkah_4(x, v, v_0)

print('=' * 62)
print('Step 4 : Hitung keluaran hidden layer')
print('=' * 62)
display(Math(r'z\_in_j = v_{0j} + \sum_{i=1}^{n} x_i v_{ij} \qquad z_j = f(z\_in_j) = \frac{1}{1 + e^{-z\_in_j}}'))
print('Keterangan: tiap baris tabel = satu node hidden (j = 1 sampai 100), n = 1024 pixel')
print('  z dekat 1 berarti node hidden aktif, dekat 0 berarti tidak aktif')
print()
print('Pengerjaan untuk node hidden 1 (j = 1):')
suku = x * v[:, 0]                       # suku ke-i: x_i x v_i1
jumlah = np.sum(suku)                    # jumlah 1024 suku
display(Math(r'z\_in_1 = v_{01} + x_1 v_{1,1} + x_2 v_{2,1} + x_3 v_{3,1} + \dots + x_{1024} v_{1024,1}'))
display(Math(rf'z\_in_1 = {v_0[0]:.4f} + ({x[0]:.4f})({v[0, 0]:.4f}) + ({x[1]:.4f})({v[1, 0]:.4f}) + ({x[2]:.4f})({v[2, 0]:.4f}) + \dots + ({x[-1]:.4f})({v[-1, 0]:.4f})'))
display(Math(rf'z\_in_1 = {v_0[0]:.4f} + ({suku[0]:.4f}) + ({suku[1]:.4f}) + ({suku[2]:.4f}) + \dots + ({suku[-1]:.4f})'))
display(Math(rf'z\_in_1 = {v_0[0]:.4f} + ({jumlah:.4f}) = {z_in[0]:.4f}'))
display(Math(rf'z_1 = f({z_in[0]:.4f}) = \frac{{1}}{{1 + e^{{-({z_in[0]:.4f})}}}} = \frac{{1}}{{1 + {np.exp(-z_in[0]):.4f}}} = {z[0]:.4f}'))
print()
print('Hasil untuk seluruh node hidden:')
print(f'{"j":>4}{"z_in":>11}{"z":>9}')
for j in range(p):
    print(f'{j + 1:>4}{z_in[j]:>+11.4f}{z[j]:>9.4f}')

## Sel 7. Step 5: Keluaran *Output Layer*

$$y\_in_k = w_{0k} + \sum_{j=1}^{p} z_j w_{jk}, \qquad y_k = f(y\_in_k)$$

* Fungsi `langkah_5` mengerjakan rumus di atas untuk tiap *node output* $k = 1, 2, 3$. Hasilnya $y_1, y_2, y_3$ adalah jawaban jaringan.
* **Analogi:** tiga juru bicara (Pesawat, Mobil, Kapal) mendengar lampu 100 ahli, menimbangnya dengan tingkat percaya masing-masing, lalu menyalakan lampu jawabannya.

In [ ]:
def langkah_5(z, w, w_0):
    """Step 5: y_in_k = w_0k + SUM_j z_j w_jk, lalu y_k = f(y_in_k)."""
    y_in = np.zeros(m)
    y    = np.zeros(m)
    for k in range(m):
        y_in[k] = w_0[k] + np.sum(z * w[:, k])
        y[k]    = f(y_in[k])
    return y_in, y


y_in, y = langkah_5(z, w, w_0)

print('=' * 62)
print('Step 5 : Hitung keluaran output layer')
print('=' * 62)
display(Math(r'y\_in_k = w_{0k} + \sum_{j=1}^{p} z_j w_{jk} \qquad y_k = f(y\_in_k)'))
print('Keterangan: tiap baris = satu node output (satu kelas), p = 100 node hidden')
print('  y adalah jawaban jaringan: dekat 1 berarti jaringan memilih kelas ini')
print()
for k in range(m):
    kelas = NAMA_KELAS[k].split('_')[1]
    suku = z * w[:, k]                   # suku ke-j: z_j x w_jk
    jumlah = np.sum(suku)                # jumlah 100 suku
    print(f'Pengerjaan untuk node output {k + 1} (k = {k + 1}, kelas {kelas}):')
    display(Math(rf'y\_in_{k + 1} = w_{{0,{k + 1}}} + z_1 w_{{1,{k + 1}}} + z_2 w_{{2,{k + 1}}} + z_3 w_{{3,{k + 1}}} + \dots + z_{{100}} w_{{100,{k + 1}}}'))
    display(Math(rf'y\_in_{k + 1} = {w_0[k]:.4f} + ({z[0]:.4f})({w[0, k]:.4f}) + ({z[1]:.4f})({w[1, k]:.4f}) + ({z[2]:.4f})({w[2, k]:.4f}) + \dots + ({z[-1]:.4f})({w[-1, k]:.4f})'))
    display(Math(rf'y\_in_{k + 1} = {w_0[k]:.4f} + ({suku[0]:.4f}) + ({suku[1]:.4f}) + ({suku[2]:.4f}) + \dots + ({suku[-1]:.4f})'))
    display(Math(rf'y\_in_{k + 1} = {w_0[k]:.4f} + ({jumlah:.4f}) = {y_in[k]:.4f}'))
    display(Math(rf'y_{k + 1} = f({y_in[k]:.4f}) = \frac{{1}}{{1 + e^{{-({y_in[k]:.4f})}}}} = \frac{{1}}{{1 + {np.exp(-y_in[k]):.4f}}} = {y[k]:.4f}'))
    print()
print('Angka z_1 sampai z_100 sama untuk ketiga node (hasil Step 4).')
print('Yang berbeda hanya bobot w_jk dan bias w_0k: tiap node output punya satu kolom bobot sendiri.')
print('Rangkuman ketiga node output:')
print(f'{"k":>4}{"Kelas":>10}{"y_in":>11}{"y":>9}')
for k in range(m):
    print(f'{k + 1:>4}{NAMA_KELAS[k].split("_")[1]:>10}{y_in[k]:>+11.4f}{y[k]:>9.4f}')
print()
print(f'Tebakan jaringan saat ini : {NAMA_KELAS[int(np.argmax(y))].split("_")[1]} (y terbesar)')
print(f'Kunci jawaban (t)         : {NAMA_KELAS[int(np.argmax(t))].split("_")[1]}')
print('Selisih y dan t inilah yang dikoreksi pada Step 6 sampai 8.')

## Sel 8. Step 6: Galat dan Koreksi di *Output Layer*

$$\delta_k = (t_k - y_k)\,f'(y\_in_k), \qquad \Delta w_{jk} = \alpha\,\delta_k\,z_j, \qquad \Delta w_{0k} = \alpha\,\delta_k$$

* Fungsi `langkah_6` membandingkan $y_k$ dengan target $t_k$, lalu menghitung faktor koreksi galat $\delta_k$ dan koreksi bobot $W$ serta *bias*-nya.
* **Analogi:** pelatih membuka kunci jawaban (lampu yang benar harus terang, sisanya padam), membandingkannya dengan lampu tiap juru bicara, lalu menegur sebesar kesalahannya.

In [ ]:
def langkah_6(t, y, y_in, z):
    """Step 6: delta_k = (t_k - y_k) f'(y_in_k); delta_w_jk = alpha delta_k z_j; delta_w_0k = alpha delta_k."""
    delta_k   = np.zeros(m)
    delta_w   = np.zeros((p, m))
    delta_w_0 = np.zeros(m)
    for k in range(m):
        delta_k[k]   = (t[k] - y[k]) * f_aksen(y_in[k])
        delta_w_0[k] = alpha * delta_k[k]
        for j in range(p):
            delta_w[j, k] = alpha * delta_k[k] * z[j]
    return delta_k, delta_w, delta_w_0


delta_k, delta_w, delta_w_0 = langkah_6(t, y, y_in, z)

print('=' * 62)
print('Step 6 : Hitung galat dan koreksi di output layer')
print('=' * 62)
display(Math(r"\delta_k = (t_k - y_k)\, f'(y\_in_k)"))
display(Math(r'\Delta w_{jk} = \alpha\, \delta_k\, z_j \qquad \Delta w_{0k} = \alpha\, \delta_k'))
print('Keterangan: tiap baris = satu node output (satu kelas)')
print('  delta_k adalah faktor koreksi galat node output')
print('  t - y positif: y kurang dari target, bobot akan dinaikkan. Negatif: y melebihi target, bobot diturunkan')
print()
for k in range(m):
    kelas = NAMA_KELAS[k].split('_')[1]
    fa = f_aksen(y_in[k])
    print(f'Pengerjaan untuk node output {k + 1} (kelas {kelas}), target t_{k + 1} = {t[k]:.0f}:')
    display(Math(rf"\delta_{k + 1} = (t_{k + 1} - y_{k + 1})\, f'(y\_in_{k + 1}) = ({t[k]:.0f} - {y[k]:.4f})\, f'({y_in[k]:.4f})"))
    display(Math(rf"f'({y_in[k]:.4f}) = f({y_in[k]:.4f})\,[1 - f({y_in[k]:.4f})] = ({y[k]:.4f})(1 - {y[k]:.4f}) = {fa:.4f}"))
    display(Math(rf"\delta_{k + 1} = ({t[k] - y[k]:.4f})({fa:.4f}) = {delta_k[k]:.5f}"))
    display(Math(rf"\Delta w_{{0,{k + 1}}} = \alpha\, \delta_{k + 1} = ({alpha})({delta_k[k]:.5f}) = {delta_w_0[k]:.6f}"))
    display(Math(rf"\Delta w_{{1,{k + 1}}} = \alpha\, \delta_{k + 1}\, z_1 = ({alpha})({delta_k[k]:.5f})({z[0]:.4f}) = {delta_w[0, k]:.6f}"))
    display(Math(rf"\Delta w_{{100,{k + 1}}} = \alpha\, \delta_{k + 1}\, z_{{100}} = ({alpha})({delta_k[k]:.5f})({z[-1]:.4f}) = {delta_w[-1, k]:.6f}"))
    print()
print('Rangkuman ketiga node output:')
print(f'{"k":>4}{"Kelas":>10}{"t":>4}{"y":>9}{"t - y":>10}{"delta_k":>11}{"delta_w_0k":>13}')
for k in range(m):
    print(f'{k + 1:>4}{NAMA_KELAS[k].split("_")[1]:>10}{t[k]:>4.0f}{y[k]:>9.4f}{t[k] - y[k]:>+10.4f}{delta_k[k]:>+11.5f}{delta_w_0[k]:>+13.5f}')
print(f'\ndelta_w berukuran {delta_w.shape}: satu koreksi untuk tiap bobot W ({p} node hidden x {m} node output)')

## Sel 9. Step 7: Galat dan Koreksi di *Hidden Layer*

$$\delta\_in_j = \sum_{k=1}^{m} \delta_k w_{jk}, \qquad \delta_j = \delta\_in_j\,f'(z\_in_j), \qquad \Delta v_{ij} = \alpha\,\delta_j\,x_i, \qquad \Delta v_{0j} = \alpha\,\delta_j$$

* Fungsi `langkah_7` mengirim $\delta_k$ mundur ke tiap *node hidden*, lalu menghitung $\delta_j$ dan koreksi bobot $V$ serta *bias*-nya. Bobot $w_{jk}$ yang dipakai masih nilai lama, karena pembaruan baru dikerjakan di Step 8.
* **Analogi:** teguran para juru bicara diteruskan mundur ke para ahli, sebanding dengan seberapa besar pengaruh tiap ahli pada jawaban juru bicara.

In [ ]:
def langkah_7(delta_k, w, z_in, x):
    """Step 7: delta_in_j = SUM_k delta_k w_jk; delta_j = delta_in_j f'(z_in_j); delta_v_ij = alpha delta_j x_i."""
    delta_in_j = np.zeros(p)
    delta_j    = np.zeros(p)
    delta_v    = np.zeros((n, p))
    delta_v_0  = np.zeros(p)
    for j in range(p):
        delta_in_j[j] = np.sum(delta_k * w[j, :])
        delta_j[j]    = delta_in_j[j] * f_aksen(z_in[j])
        delta_v_0[j]  = alpha * delta_j[j]
        delta_v[:, j] = alpha * delta_j[j] * x     # delta_v_ij untuk semua i sekaligus
    return delta_in_j, delta_j, delta_v, delta_v_0


delta_in_j, delta_j, delta_v, delta_v_0 = langkah_7(delta_k, w, z_in, x)

print('=' * 62)
print('Step 7 : Hitung galat dan koreksi di hidden layer')
print('=' * 62)
display(Math(r"\delta\_in_j = \sum_{k=1}^{m} \delta_k w_{jk} \qquad \delta_j = \delta\_in_j\, f'(z\_in_j)"))
display(Math(r'\Delta v_{ij} = \alpha\, \delta_j\, x_i \qquad \Delta v_{0j} = \alpha\, \delta_j'))
print('Keterangan: tiap baris = satu node hidden (j = 1 sampai 100), m = 3 node output')
print('  delta_in_j adalah teguran dari 3 node output yang sampai ke node hidden j')
print('  delta_j adalah faktor koreksi galat node hidden j')
print()
print('Pengerjaan untuk node hidden 1 (j = 1):')
suku = delta_k * w[0, :]                 # suku ke-k: delta_k x w_1k
fa = f_aksen(z_in[0])
display(Math(r"\delta\_in_1 = \sum_{k=1}^{3} \delta_k w_{1,k} = \delta_1 w_{1,1} + \delta_2 w_{1,2} + \delta_3 w_{1,3}"))
display(Math(rf"\delta\_in_1 = ({delta_k[0]:.5f})({w[0, 0]:.4f}) + ({delta_k[1]:.5f})({w[0, 1]:.4f}) + ({delta_k[2]:.5f})({w[0, 2]:.4f})"))
display(Math(rf"\delta\_in_1 = ({suku[0]:.6f}) + ({suku[1]:.6f}) + ({suku[2]:.6f}) = {delta_in_j[0]:.6f}"))
display(Math(rf"f'(z\_in_1) = f({z_in[0]:.4f})\,[1 - f({z_in[0]:.4f})] = ({z[0]:.4f})(1 - {z[0]:.4f}) = {fa:.4f}"))
display(Math(rf"\delta_1 = \delta\_in_1\, f'(z\_in_1) = ({delta_in_j[0]:.6f})({fa:.4f}) = {delta_j[0]:.6f}"))
display(Math(rf"\Delta v_{{0,1}} = \alpha\, \delta_1 = ({alpha})({delta_j[0]:.6f}) = {delta_v_0[0]:.7f}"))
display(Math(rf"\Delta v_{{1,1}} = \alpha\, \delta_1\, x_1 = ({alpha})({delta_j[0]:.6f})({x[0]:.4f}) = {delta_v[0, 0]:.7f}"))
display(Math(rf"\Delta v_{{1024,1}} = \alpha\, \delta_1\, x_{{1024}} = ({alpha})({delta_j[0]:.6f})({x[-1]:.4f}) = {delta_v[-1, 0]:.7f}"))
print('Node hidden 2 sampai 100 dikerjakan dengan cara yang sama. Hasilnya ada di tabel berikut.')
print()
print(f'{"j":>4}{"delta_in_j":>13}{"delta_j":>12}{"delta_v_0j":>13}')
for j in range(p):
    print(f'{j + 1:>4}{delta_in_j[j]:>+13.5f}{delta_j[j]:>+12.6f}{delta_v_0[j]:>+13.6f}')
print(f'\ndelta_v berukuran {delta_v.shape}: satu koreksi untuk tiap bobot V ({n} pixel x {p} node hidden)')

## Sel 10. Step 8: Perbarui Bobot dan *Bias*

$$w_{jk}(\text{new}) = w_{jk}(\text{old}) + \Delta w_{jk}, \qquad v_{ij}(\text{new}) = v_{ij}(\text{old}) + \Delta v_{ij}$$

* Fungsi `langkah_8` menambahkan semua koreksi dari Step 6 dan 7 ke bobot dan *bias* (rumus yang sama berlaku untuk $w_{0k}$ dan $v_{0j}$).
* Contoh di sel ini memakai **salinan** bobot, sehingga bobot asli belum berubah sebelum Sel 11.
* **Analogi:** semua orang mencatat tegurannya, lalu mengubah tingkat percaya dan kecenderungannya: baru = lama + koreksi. Kartu foto berikutnya dikerjakan dengan catatan yang baru.

In [ ]:
def langkah_8(v, v_0, w, w_0, delta_v, delta_v_0, delta_w, delta_w_0):
    """Step 8: bobot baru = bobot lama + koreksi. Larik v, v_0, w, w_0 diubah langsung (in place)."""
    for k in range(m):
        w_0[k]  = w_0[k] + delta_w_0[k]
        w[:, k] = w[:, k] + delta_w[:, k]          # semua j sekaligus
    for j in range(p):
        v_0[j]  = v_0[j] + delta_v_0[j]
        v[:, j] = v[:, j] + delta_v[:, j]          # semua i sekaligus


# salinan: bobot asli tidak berubah
v_s, v_0_s, w_s, w_0_s = v.copy(), v_0.copy(), w.copy(), w_0.copy()
langkah_8(v_s, v_0_s, w_s, w_0_s, delta_v, delta_v_0, delta_w, delta_w_0)

print('=' * 62)
print('Step 8 : Perbarui bobot dan bias')
print('=' * 62)
display(Math(r'w_{jk}(\mathrm{new}) = w_{jk}(\mathrm{old}) + \Delta w_{jk} \qquad v_{ij}(\mathrm{new}) = v_{ij}(\mathrm{old}) + \Delta v_{ij}'))
print('Keterangan: bobot baru = bobot lama + koreksi (dari Step 6 dan 7)')
print('  Semua 102.803 bobot dan bias dikoreksi dengan cara yang sama. Berikut pengerjaan beberapa di antaranya.')
print()
print('Bobot W (hidden 1 ke tiap node output) dan bias W:')
for k in range(m):
    kelas = NAMA_KELAS[k].split('_')[1]
    display(Math(rf'w_{{1,{k + 1}}}(\mathrm{{new}}) = w_{{1,{k + 1}}}(\mathrm{{old}}) + \Delta w_{{1,{k + 1}}} = ({w[0, k]:.5f}) + ({delta_w[0, k]:+.6f}) = {w_s[0, k]:.5f} \quad \text{{({kelas})}}'))
for k in range(m):
    display(Math(rf'w_{{0,{k + 1}}}(\mathrm{{new}}) = w_{{0,{k + 1}}}(\mathrm{{old}}) + \Delta w_{{0,{k + 1}}} = ({w_0[k]:.5f}) + ({delta_w_0[k]:+.6f}) = {w_0_s[k]:.5f}'))
print()
print('Bobot V (ke hidden 1) dan bias V:')
display(Math(rf'v_{{1,1}}(\mathrm{{new}}) = v_{{1,1}}(\mathrm{{old}}) + \Delta v_{{1,1}} = ({v[0, 0]:.5f}) + ({delta_v[0, 0]:+.7f}) = {v_s[0, 0]:.5f}'))
display(Math(rf'v_{{1024,1}}(\mathrm{{new}}) = v_{{1024,1}}(\mathrm{{old}}) + \Delta v_{{1024,1}} = ({v[-1, 0]:.5f}) + ({delta_v[-1, 0]:+.7f}) = {v_s[-1, 0]:.5f}'))
display(Math(rf'v_{{0,1}}(\mathrm{{new}}) = v_{{0,1}}(\mathrm{{old}}) + \Delta v_{{0,1}} = ({v_0[0]:.5f}) + ({delta_v_0[0]:+.7f}) = {v_0_s[0]:.5f}'))
print()
print('Bobot baru inilah yang dipakai pada pasangan latih berikutnya (Step 2).')


## Sel 11. Step 1, 2, dan 9: Siklus Pelatihan

* Step 1: ulangi selama syarat berhenti belum terpenuhi. Step 2: untuk setiap pasangan latih $(x, t)$, kerjakan Step 3 sampai 8 (fungsi dari Sel 6 sampai 10), dengan urutan pasangan diacak tiap *epoch*.
* Step 9: uji syarat berhenti, yaitu galat rata-rata di bawah ambang atau *epoch* maksimum tercapai. Bobot diperbarui per pasangan latih, seperti di buku.
* *Output* menampilkan label Step 0, 1, 2, dan 9. Tabel dicetak pada *epoch* 1 dan tiap 10 *epoch*.
* **Analogi:** pelatih mengulang langkah di atas untuk 900 kartu foto (satu putaran = satu *epoch*), kartunya diacak. Setelah satu putaran, pelatih menghitung rata-rata kesalahan: bila sudah di bawah ambang, atau putaran maksimum tercapai, latihan selesai. Bila belum, ulangi.

In [ ]:
def umpan_maju(X):
    """Step 4 dan 5 untuk banyak pola, tanpa mengubah bobot. Hasil: matriks y (baris = pola)."""
    hasil = np.zeros((len(X), m))
    for s in range(len(X)):
        z_in, z = langkah_4(X[s], v, v_0)
        y_in, y = langkah_5(z, w, w_0)
        hasil[s] = y
    return hasil

def akurasi(X, label):
    # argmax: nomor node output terbesar = kelas tebakan
    return np.mean(np.argmax(umpan_maju(X), axis=1) == label) * 100


riwayat_galat, riwayat_akurasi = [], []   # catatan tiap epoch, untuk grafik
jumlah_pola = len(X_latih)

print('=' * 78)
print('Step 0 : Initialize weights (Sel 4)')
print('Step 1 : While stopping condition is false, do Steps 2 to 9')
print(f'Step 2 : For each training pair, do Steps 3 to 8  ({jumlah_pola} pasangan, urutan diacak)')
print(f'Step 9 : Test stopping condition (galat rata-rata < {AMBANG_GALAT} atau epoch = {EPOCH_MAKS})')
print('=' * 78)
print('Keterangan kolom:')
print('  Epoch          : putaran ke berapa (satu epoch = seluruh pasangan latih dikerjakan sekali)')
print('  Step 2         : jumlah pasangan latih yang dikerjakan pada epoch itu')
print('  Step 9 galat   : rata-rata kesalahan per pola, makin kecil makin baik')
print('  Akurasi latih  : persen gambar latih yang ditebak benar')
print('=' * 78)
print(f'{"Epoch":>6}{"Step 2: pasangan":>18}{"Step 9: galat":>15}{"Akurasi latih":>15}   Syarat berhenti')
print('-' * 78)

waktu_mulai = time.time()

for epoch in range(1, EPOCH_MAKS + 1):                      # Step 1
    galat_total = 0.0
    for s in rng.permutation(jumlah_pola):                  # Step 2
        x = X_latih[s]                                      # Step 3
        t = T_latih[s]
        z_in, z                         = langkah_4(x, v, v_0)               # Step 4
        y_in, y                         = langkah_5(z, w, w_0)               # Step 5
        delta_k, delta_w, delta_w_0     = langkah_6(t, y, y_in, z)           # Step 6
        delta_in_j, delta_j, delta_v, delta_v_0 = langkah_7(delta_k, w, z_in, x)   # Step 7
        langkah_8(v, v_0, w, w_0, delta_v, delta_v_0, delta_w, delta_w_0)    # Step 8
        galat_total += 0.5 * np.sum((t - y) ** 2)           # galat satu pola

    galat_rata = galat_total / jumlah_pola
    riwayat_galat.append(galat_rata)
    riwayat_akurasi.append(akurasi(X_latih, label_latih))

    # Step 9
    if galat_rata < AMBANG_GALAT:
        keputusan = 'ya, galat di bawah ambang'
    elif epoch == EPOCH_MAKS:
        keputusan = 'ya, epoch maksimum tercapai'
    else:
        keputusan = 'belum, kembali ke Step 1'
    berhenti = keputusan.startswith('ya')

    if epoch == 1 or epoch % CETAK_TIAP == 0 or berhenti:
        print(f'{epoch:>6}{jumlah_pola:>18}{galat_rata:>15.5f}{riwayat_akurasi[-1]:>14.1f}%   {keputusan}')
    if epoch == 1:
        print(f'       (waktu 1 epoch: {time.time() - waktu_mulai:.1f} detik)')
    if berhenti:
        break

print('-' * 78)
print(f'Epoch dijalankan : {len(riwayat_galat)}')
print(f'Galat akhir      : {riwayat_galat[-1]:.5f}')
print(f'Akurasi latih    : {riwayat_akurasi[-1]:.1f}%')
print(f'Waktu total      : {time.time() - waktu_mulai:.1f} detik')

## Sel 12. Grafik Hasil Belajar

* Menggambar rekaman galat rata-rata dan akurasi data latih untuk setiap *epoch* dari pelatihan tadi, sehingga penurunan galat dan kenaikan akurasi dapat dilihat tanpa komputasi ulang.
* **Analogi:** pelatih mencatat nilai tim tiap putaran. Bila latihan berjalan baik, kesalahan makin turun dan ketepatan makin naik.

In [ ]:
# kiri: galat tiap epoch (diharapkan turun); kanan: akurasi data latih (diharapkan naik)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
a1.plot(range(1, len(riwayat_galat) + 1), riwayat_galat)
a1.set_title('Galat rata-rata per epoch'); a1.set_xlabel('Epoch'); a1.set_ylabel('Galat'); a1.grid(alpha=0.3)
a2.plot(range(1, len(riwayat_akurasi) + 1), riwayat_akurasi, color='tab:green')
a2.set_title('Akurasi data latih (%)'); a2.set_xlabel('Epoch'); a2.set_ylim(0, 100); a2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Sel 13. Evaluasi Model dan *Confusion Matrix*

* Menguji performa model pada data latih dan data uji, lalu membandingkan keduanya untuk melihat indikasi *overfitting* (model hafal data latih: akurasi latih jauh di atas akurasi uji).
* Menyajikan tabel tebakan (*confusion matrix*): baris adalah kelas sebenarnya dan kolom adalah tebakan model, sehingga angka di diagonal adalah tebakan tepat dan angka di luar diagonal adalah salah tebak tiap kelas.
* **Analogi:** ujian akhir dengan 150 foto yang belum pernah dilihat tim. Bila nilai latihan nyaris sempurna tetapi nilai ujian jauh lebih rendah, tim hanya menghafal kartu latihan (*overfitting*). Tabel tebakan adalah rapor per jenis kendaraan.

In [ ]:
def laporan(X, label, judul):
    tebakan = np.argmax(umpan_maju(X), axis=1)   # tebakan: nomor node output bernilai terbesar
    benar = np.mean(tebakan == label) * 100
    print(f'{judul}: {benar:.1f}%  ({int((tebakan == label).sum())} dari {len(label)} gambar benar)')
    return tebakan

print('=' * 52)
print('EVALUASI')
print('=' * 52)
laporan(X_latih, label_latih, 'Akurasi data latih')
tebakan_uji = laporan(X_uji, label_uji, 'Akurasi data uji  ')

# confusion matrix: baris = kelas sebenarnya, kolom = tebakan model
print('\nTabel tebakan pada data uji (baris = kelas sebenarnya, kolom = tebakan jaringan, angka diagonal = tebakan benar)')
judul = [nama.split('_')[1] for nama in NAMA_KELAS]
print(f'{"Sebenarnya":<12}' + ''.join(f'{j:>9}' for j in judul) + f'{"Akurasi":>10}')
for k in range(m):
    baris = [int(np.sum((label_uji == k) & (tebakan_uji == j))) for j in range(m)]
    persen = baris[k] / np.sum(label_uji == k) * 100
    print(f'{judul[k]:<12}' + ''.join(f'{b:>9}' for b in baris) + f'{persen:>9.0f}%')

## Sel 14. Inspeksi Keluaran Jaringan terhadap Target

* Menampilkan perbandingan nilai keluaran jaringan ($Y_1, Y_2, Y_3$) terhadap target pada sampel data uji (3 gambar tiap kelas) untuk memeriksa ketegasan tebakan model: node yang benar seharusnya mendekati 1, sisanya mendekati 0.
* **Analogi:** melihat langsung tiga lampu juru bicara pada beberapa foto uji. Lampu paling terang adalah jawaban tim, dan jawaban benar bila sama dengan lampu pada kunci jawaban.

In [ ]:
print('=' * 62)
print('POLA TARGET OUTPUT (t)')
print('=' * 62)
print(f'{"Kelas":<18}{"Y1":>4}{"Y2":>4}{"Y3":>4}   Artinya')
for k in range(m):
    pola = ''.join(f'{int(a):>4}' for a in np.eye(m)[k])
    print(f'{NAMA_KELAS[k]:<18}{pola}   hanya Y{k + 1} bernilai 1')

print()
print('=' * 62)
print('KELUARAN JARINGAN (y) DIBANDINGKAN DENGAN TARGET (t), DATA UJI')
print('=' * 62)
print(f'{"Kelas":<16}{"Target t":<10}{"Keluaran y (Y1, Y2, Y3)":<26}{"Hasil"}')
print('-' * 62)
y_uji = umpan_maju(X_uji)   # keluaran jaringan (y) untuk seluruh data uji
for k in range(m):
    for s in np.where(label_uji == k)[0][:3]:
        t_teks = ' '.join(str(int(a)) for a in T_uji[s])
        y_teks = '  '.join(f'{a:.3f}' for a in y_uji[s])
        # BENAR bila node dengan nilai terbesar adalah node kelas sebenarnya
        hasil = 'BENAR' if np.argmax(y_uji[s]) == k else 'SALAH'
        print(f'{NAMA_KELAS[k].split("_")[1]:<16}{t_teks:<10}{y_teks:<26}{hasil}')

## Sel 15. Penyimpanan Parameter Bobot Terlatih

* Menyimpan matriks bobot final ($v, v_0, w, w_0$) beserta ukuran gambar ke dalam satu berkas `BOBOT.npz` (format NumPy untuk beberapa larik dalam satu berkas) di Google Drive, lalu memverifikasi bahwa isi berkas sama dengan bobot di memori.
* **Analogi:** pelatih menulis semua tingkat percaya dan kecenderungan ke satu buku catatan (`BOBOT.npz`). Pada program pengujian, tim cukup membaca buku itu tanpa belajar dari awal.

In [ ]:
os.makedirs(os.path.dirname(BERKAS_BOBOT), exist_ok=True)   # buat folder BOBOT bila belum ada
# savez: simpan beberapa larik NumPy dalam satu berkas .npz
# ukuran gambar ikut disimpan agar program pengujian memakai ukuran yang sama
np.savez(BERKAS_BOBOT, v=v, v_0=v_0, w=w, w_0=w_0, ukuran=np.array([LEBAR, TINGGI]))

# Pemeriksaan: baca ulang berkas yang baru disimpan
cek = np.load(BERKAS_BOBOT)
print('=' * 52)
print('BOBOT TERSIMPAN')
print('=' * 52)
print(f'Lokasi : {BERKAS_BOBOT}')
print(f'Ukuran : {os.path.getsize(BERKAS_BOBOT) / 1024:.0f} KB\n')
print(f'{"Isi":<10}{"Ukuran":<14}')
for kunci in ['v', 'v_0', 'w', 'w_0']:
    print(f'{kunci:<10}{str(cek[kunci].shape):<14}')
sama = all(np.array_equal(cek[k], a) for k, a in [('v', v), ('v_0', v_0), ('w', w), ('w_0', w_0)])
print(f'\nIsi berkas sama dengan bobot di memori: {"YA" if sama else "TIDAK"}')

## Sel 16. Versi ZIP Bobot yang Bisa Dibaca dan Unduhan Otomatis

* Membuat `BOBOT.zip` berisi 5 berkas Excel (`.xlsx`), satu untuk tiap isi `BOBOT.npz`: `v.xlsx`, `v_0.xlsx`, `w.xlsx`, `w_0.xlsx`, dan `ukuran.xlsx`. Tiap berkas punya judul kolom dan baris yang tebal, kolom yang cukup lebar, garis tepi (*outline*) pada semua sel, serta baris dan kolom judul yang tetap terlihat saat digulir.
* Angka disimpan utuh sebagai bilangan dan hanya ditampilkan 6 desimal, sehingga tampilannya mengikuti pengaturan Excel masing-masing (titik atau koma desimal).
* Membuka kembali `BOBOT.zip`, membaca kelima berkas, lalu memeriksa bahwa nilainya sama dengan bobot di memori (selisih di bawah 1e-12, karena Excel menyimpan sekitar 15 digit). Bobot utuh tetap ada di `BOBOT.npz`.
* Menyimpan `BOBOT.zip` di Google Drive dan mengunduh `BOBOT.npz` serta `BOBOT.zip` ke komputer. Bila peramban menanyakan izin unduh beberapa berkas, pilih izinkan.
* Program pengujian tetap membaca `BOBOT.npz`. Berkas ZIP hanya untuk dibaca manusia.

In [ ]:
import io
import zipfile
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill
from openpyxl.utils import get_column_letter
from google.colab import files

BERKAS_ZIP = os.path.dirname(BERKAS_BOBOT) + '/BOBOT.zip'
nama_kelas = [k.split('_')[1] for k in NAMA_KELAS]

garis    = Side(style='thin', color='000000')
tepi     = Border(left=garis, right=garis, top=garis, bottom=garis)
isi_judul = PatternFill('solid', fgColor='D9E1F2')
huruf_tebal = Font(name='Calibri', bold=True)
rata_tengah = Alignment(horizontal='center', vertical='center')
rata_kanan  = Alignment(horizontal='right')

def buat_xlsx(judul, baris, nama_sheet):
    """judul: daftar judul kolom (kolom pertama = judul label baris). baris: daftar baris, elemen pertama = label."""
    wb = Workbook()
    ws = wb.active
    ws.title = nama_sheet
    ws.append(judul)
    for b in baris:
        ws.append(b)
    for c in ws[1]:                                   # baris judul
        c.font, c.fill, c.alignment, c.border = huruf_tebal, isi_judul, rata_tengah, tepi
    for r in ws.iter_rows(min_row=2):
        r[0].font, r[0].fill, r[0].alignment, r[0].border = huruf_tebal, isi_judul, rata_tengah, tepi
        for c in r[1:]:
            c.border = tepi
            if isinstance(c.value, float):
                c.number_format = '0.000000'          # tampil 6 desimal, nilai tetap utuh
                c.alignment = rata_kanan
            else:
                c.alignment = rata_tengah
    for k, kepala in enumerate(judul, start=1):       # kolom cukup lebar
        ws.column_dimensions[get_column_letter(k)].width = max(14, len(str(kepala)) + 4)
    ws.freeze_panes = 'B2'                            # judul tetap terlihat saat digulir
    ws.row_dimensions[1].height = 22
    penampung = io.BytesIO()
    wb.save(penampung)
    return penampung.getvalue()

isi_zip = {
    'v.xlsx'     : buat_xlsx(['i \\ j'] + [f'j={j + 1}' for j in range(p)],
                             [[f'i={i + 1}'] + [float(a) for a in v[i]] for i in range(n)], 'v'),
    'v_0.xlsx'   : buat_xlsx(['j', 'v_0j'],
                             [[f'j={j + 1}', float(v_0[j])] for j in range(p)], 'v_0'),
    'w.xlsx'     : buat_xlsx(['j \\ k'] + [f'k={k + 1} ({nama_kelas[k]})' for k in range(m)],
                             [[f'j={j + 1}'] + [float(a) for a in w[j]] for j in range(p)], 'w'),
    'w_0.xlsx'   : buat_xlsx(['k', 'w_0k', 'Kelas'],
                             [[f'k={k + 1}', float(w_0[k]), nama_kelas[k]] for k in range(m)], 'w_0'),
    'ukuran.xlsx': buat_xlsx(['Besaran', 'Nilai'],
                             [['Lebar', LEBAR], ['Tinggi', TINGGI]], 'ukuran'),
}
with zipfile.ZipFile(BERKAS_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for nama, isi in isi_zip.items():
        zf.writestr(nama, isi)

# Pemeriksaan: buka kembali ZIP, baca kelima berkas, bandingkan dengan bobot di memori
def baca_xlsx(isi):
    ws = load_workbook(io.BytesIO(isi), data_only=True).active
    return [[sel.value for sel in baris] for baris in ws.iter_rows(min_row=2)]

with zipfile.ZipFile(BERKAS_ZIP) as zf:
    assert zf.testzip() is None
    daftar = zf.namelist()
    ukuran_berkas = {nm: zf.getinfo(nm).file_size for nm in daftar}
    isi_baca = {nm: baca_xlsx(zf.read(nm)) for nm in daftar}

hasil = {
    'v.xlsx'     : np.allclose(np.array([b[1:] for b in isi_baca['v.xlsx']], dtype=float), v, rtol=0, atol=1e-12),
    'v_0.xlsx'   : np.allclose(np.array([b[1] for b in isi_baca['v_0.xlsx']], dtype=float), v_0, rtol=0, atol=1e-12),
    'w.xlsx'     : np.allclose(np.array([b[1:] for b in isi_baca['w.xlsx']], dtype=float), w, rtol=0, atol=1e-12),
    'w_0.xlsx'   : np.allclose(np.array([b[1] for b in isi_baca['w_0.xlsx']], dtype=float), w_0, rtol=0, atol=1e-12),
    'ukuran.xlsx': [b[1] for b in isi_baca['ukuran.xlsx']] == [LEBAR, TINGGI],
}

print('=' * 52)
print('BOBOT.zip')
print('=' * 52)
print(f'Lokasi : {BERKAS_ZIP}')
print(f'Ukuran : {os.path.getsize(BERKAS_ZIP) / 1024:.0f} KB\n')
print(f'{"Berkas":<14}{"Ukuran":>10}   Sama dengan memori (selisih < 1e-12)')
for nm in daftar:
    print(f'{nm:<14}{ukuran_berkas[nm] / 1024:>7.0f} KB   {"YA" if hasil[nm] else "TIDAK"}')
print(f'\nJumlah berkas di dalam ZIP: {len(daftar)}')

print('\nContoh isi w.xlsx (baris dan kolom judul, 4 baris pertama):')
print(f'{"":<8}' + ''.join(f'{kepala:>18}' for kepala in [f'k={k + 1} ({nama_kelas[k]})' for k in range(m)]))
for j in range(4):
    print(f'{"j=" + str(j + 1):<8}' + ''.join(f'{w[j, k]:>18.6f}' for k in range(m)))

files.download(BERKAS_BOBOT)
files.download(BERKAS_ZIP)